# La Fossa volcano (Vulcano Island, Italy) — from magma compressibility to surface uplift
### Magma compressibility → chamber volume change → surface displacement (Mogi / Yang / Fialko penny-shaped crack)

This notebook runs the complete seven-step chain of the thesis and **displays every result directly on this page** — input tables, the magmatic state of each reservoir, gas speciation and fugacities, compressibilities, $r_V$, volume changes, the completed `final_scheme_FREYA` uplift tables, the detectability analysis, the validation against the 2021 unrest, figures, and the complete numerical results. **No Word, Excel or CSV file is written or downloaded.**

1. **Pressure from depth:** $P=\rho g z$
2. **Gas volume fraction $\varphi$ and fugacities** at reservoir $P,T$, computed with the thermodynamic code **EVo** (closed-system C-O-H-S; Liggins et al., github.com/pipliggins/EVo) from measured Vulcano melt compositions and volatile budgets
3. **Magma compressibility:** $\beta_m=\varphi\,\beta_{gas}+(1-\varphi)\,\beta_{liquid}$, with $\beta_{gas}=1/P$
4. **Chamber compressibility $\beta_c$** from the reservoir shape (sphere, prolate spheroid, penny crack)
5. **Volume-partitioning factor:** $r_V = 1+\beta_m/\beta_c$
6. **Reservoir volume change:** $\Delta V_c = V_e / r_V$
7. **Surface uplift** from Mogi (1958), Yang et al. (1988)/Newman et al. (2006) and Fialko et al. (2001), as implemented in dMODELS (Battaglia et al. 2013, via `dmodelspy`)

**How to run:** *Runtime ▸ Run all*. The measured Vulcano data (melt compositions, temperatures, H2O–CO2–S budgets, fO2 and the Di Traglia et al. 2023 source geometry) are form levers in cells 2 and 3; the detectability target and strength limit are levers in cell 7. After changing a lever, use *Runtime ▸ Run after*. The defaults reproduce Scenario A of the thesis exactly.


In [ ]:
#@title 1) Install prerequisites (EVo + dmodelspy) and load the on-screen display toolkit  { display-mode: "form" }
#@markdown Everything this notebook produces is shown directly on this page: tables, figures and
#@markdown the completed scheme. No Word, Excel or CSV files are written.
import os, sys, subprocess, html
import numpy as np
import pandas as pd

def _sh(cmd):
    subprocess.run(cmd, shell=True, check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Versions pinned for reproducibility (the thesis results were produced with exactly these)
EVO_COMMIT = "2487939e18d98292f9b8f3a1f0dea04b707bd89f"   # pipliggins/EVo, 5 June 2026
_sh("pip -q install dmodelspy==0.2 pyyaml ruamel.yaml mpmath pydantic")
if not os.path.isdir("EVo"):
    _sh("git clone -q https://github.com/pipliggins/EVo.git")
    _sh(f"git -C EVo checkout -q {EVO_COMMIT}")

EVO_SRC = os.path.abspath("EVo/src")          # import EVo straight from its source tree
if EVO_SRC not in sys.path:
    sys.path.insert(0, EVO_SRC)
import evo
from evo import messages as _evo_msgs
_evo_msgs.query_yes_no = lambda *a, **k: True  # never block on interactive prompts
def _evo_exit(msg=None):
    raise SystemExit(msg)
# EVo stops with a bare exit(), which is not defined inside a Jupyter/Colab kernel: give every EVo
# module an exit() that raises SystemExit, so an undersaturated magma is reported instead of crashing.
for _n, _m in list(sys.modules.items()):
    if _n == "evo" or _n.startswith("evo."):
        setattr(_m, "exit", _evo_exit)
import dmodelspy
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# ---------------------------------------------------------------- display toolkit
# Colab renders each output in its own frame, so the style sheet travels with every block.
_CSS = """<style>
.lfw{font-family:Arial,Helvetica,sans-serif;color:#111;max-width:1180px}
.lfh{font-size:18px;font-weight:700;color:#fff;background:#1F4E79;padding:7px 12px;margin:16px 0 8px;border-radius:3px}
.lft{font-size:14px;font-weight:700;color:#1F4E79;margin:14px 0 5px}
.lfc{font-size:12px;color:#444;margin:5px 0 14px;line-height:1.45}
table.lf{border-collapse:collapse;font-size:12.5px;background:#fff;color:#111;margin:2px 0}
table.lf th{background:#2E5D8A;color:#fff;padding:5px 8px;border:1px solid #9fb3c8;text-align:center;font-weight:600}
table.lf td{padding:4px 8px;border:1px solid #cdd6e0;text-align:center;vertical-align:middle;background:#fff}
table.lf tr:nth-child(even) td{background:#f4f7fb}
table.lf td.l{text-align:left}
table.lf td.bad{background:#fde3e1 !important;color:#8a1c1c;font-weight:700}
table.lf td.good{background:#e4f5e7 !important;color:#15521f}
table.lf td.hi{background:#fff4d6 !important;font-weight:700}
.lfb{border-left:5px solid #2E75B6;background:#eef4fb;padding:8px 12px;margin:8px 0;font-size:13px;line-height:1.55;color:#111}
.lfb.warn{border-color:#c0392b;background:#fdf0ef}
.lfb.ok{border-color:#2e8b57;background:#edf8f1}
</style>"""

def _show(inner):
    display(HTML(_CSS + '<div class="lfw">' + inner + '</div>'))

def section(title):
    """Coloured section banner."""
    _show(f'<div class="lfh">{title}</div>')

def note(text, kind="info"):
    """Highlighted paragraph: kind = 'info', 'ok' or 'warn'."""
    cls = {"info": "lfb", "ok": "lfb ok", "warn": "lfb warn"}[kind]
    _show(f'<div class="{cls}">{text}</div>')

def table(header, rows, title=None, caption=None, classes=None, left_cols=(0,)):
    """Render a table on the page. rows = list of lists of (HTML) strings;
    classes(i, j) may return 'bad', 'good' or 'hi' to colour a cell."""
    head = "".join(f"<th>{c}</th>" for c in header)
    body = []
    for i, r in enumerate(rows):
        tds = []
        for j, v in enumerate(r):
            cl = ["l"] if j in left_cols else []
            extra = classes(i, j) if classes else None
            if extra:
                cl.append(extra)
            tds.append(f'<td class="{" ".join(cl)}">{v}</td>')
        body.append("<tr>" + "".join(tds) + "</tr>")
    out = f'<div class="lft">{title}</div>' if title else ""
    out += f'<table class="lf"><tr>{head}</tr>{"".join(body)}</table>'
    if caption:
        out += f'<div class="lfc">{caption}</div>'
    _show(out)

# ---------------------------------------------------------------- number formatting
def sci(v, nd=2):
    """1.23×10⁻⁴ style (HTML)."""
    if v is None or not np.isfinite(v):
        return "—"
    if v == 0:
        return "0"
    e = int(np.floor(np.log10(abs(v))))
    m = v / 10 ** e
    if abs(round(m, nd)) >= 10:
        m, e = m / 10, e + 1
    return f"{m:.{nd}f}×10<sup>{e}</sup>".replace("-", "−")

def num(v, nd=2):
    return "—" if v is None or not np.isfinite(v) else f"{v:.{nd}f}"

def mm(v):
    """Displacement in mm with sensible precision."""
    a = abs(v)
    if a >= 100:  return f"{v:.0f} mm"
    if a >= 1:    return f"{v:.2f} mm"
    if a >= 0.01: return f"{v:.3f} mm"
    return f"{v * 1000:.1f} µm"

def cfg_name(r):
    """Readable configuration label for a results row."""
    if r["model"] == "MOGI":
        return "Mogi sphere"
    if r["model"] == "YANG":
        return "Yang spheroid (2021 source)"
    return f"Penny crack, a = {r['penny_radius_km']:g} km"

plt.rcParams.update({"font.family": "serif", "font.size": 9.5, "axes.linewidth": 0.8,
                     "xtick.direction": "in", "ytick.direction": "in", "figure.dpi": 110})

section("Environment ready")
note(f"EVo loaded from <code>{html.escape(evo.__file__)}</code>; dmodelspy loaded. "
     "All results below will be displayed on this page.", "ok")


In [ ]:
#@title 2) Physical levers: crust, elastic moduli, fO2, and the Di Traglia (2023) source geometry  { display-mode: "form", run: "auto" }
#@markdown **Crust and pressure** &nbsp; (P = rho * g * z)
rho_crust = 2500      #@param {type:"slider", min:2200, max:2900, step:50}
g = 9.81              #@param {type:"number"}

#@markdown **Host-rock elasticity.** Deep reservoirs (2/5/12 km) sit in intact rock; the shallow 2021 source sits in damaged, hydrothermally altered rock. Both values follow Heap et al. (2020) as adopted by Di Traglia et al. (2023): 1 GPa for altered rock, 10 GPa for intact rock.
mu_crust_GPa = 10.0   #@param {type:"slider", min:5, max:40, step:1}
nu = 0.25             #@param {type:"slider", min:0.15, max:0.35, step:0.01}
mu_shallow_GPa = 1.0  #@param {type:"slider", min:0.5, max:10, step:0.5}
nu_shallow = 0.35     #@param {type:"slider", min:0.2, max:0.45, step:0.01}

#@markdown **Magma oxidation state** — fO2 relative to the FMQ buffer (oxidised arc magmas: FMQ+0.5 to +1.5)
dFMQ = 1.0            #@param {type:"slider", min:-1.0, max:2.0, step:0.25}

#@markdown **Yang source geometry (Di Traglia et al. 2023, GRL, Supporting Information Table S1, nu = 0.35).** Dipping prolate spheroid at 598 m below sea level beneath La Fossa: semi-axes 595 / 59 m (aspect ratio 0.099), dip 69 deg, azimuth 137 deg (NW-SE), V0 = 8.83e6 m3, dV = 7.31e4 m3.
yang_aspect = 0.099   #@param {type:"slider", min:0.05, max:0.99, step:0.005}
yang_dip = 69         #@param {type:"slider", min:40, max:90, step:1}
yang_strike = 137     #@param {type:"slider", min:0, max:360, step:1}
profile_azimuth = 0   #@param {type:"slider", min:0, max:360, step:15}

#@markdown **Pressure datum of the Yang source.** Di Traglia et al. give 598 m *below sea level*; the real overburden below the ground surface of the edifice is ~0.8-0.99 km. Keeping 0.598 uses the model depth itself (the conservative, gas-rich choice).
yang_overburden_km = 0.598 #@param {type:"slider", min:0.4, max:1.2, step:0.01}

#@markdown **Engine for the gas fraction.** If EVo is unavailable, a simple analytical solubility model is used instead.
use_evo = True        #@param {type:"boolean"}

PHYS = dict(rho_crust=float(rho_crust), g=float(g),
            mu_crust_GPa=float(mu_crust_GPa), mu_shallow_GPa=float(mu_shallow_GPa),
            nu=float(nu), nu_shallow=float(nu_shallow), dFMQ=float(dFMQ),
            yang_aspect=float(yang_aspect), yang_dip=float(yang_dip),
            yang_strike=float(yang_strike), profile_azimuth=float(profile_azimuth),
            yang_overburden_km=float(yang_overburden_km),
            use_evo=bool(use_evo))
section("Physical parameters in use")
table(["Parameter", "Value", "Parameter", "Value"],
      [["Crustal density ρ<sub>c</sub>", f"{rho_crust:.0f} kg m<sup>−3</sup>", "Yang aspect ratio b/a", f"{yang_aspect:.3f}"],
       ["Shear modulus, deep μ", f"{mu_crust_GPa:g} GPa", "Yang dip / azimuth", f"{yang_dip:g}° / {yang_strike:g}°"],
       ["Shear modulus, shallow μ", f"{mu_shallow_GPa:g} GPa", "Pressure datum of Yang source", f"{yang_overburden_km:.3f} km"],
       ["Poisson ratio deep / shallow", f"{nu:.2f} / {nu_shallow:.2f}", "Profile azimuth", f"{profile_azimuth:g}°"],
       ["Redox buffer", f"FMQ {dFMQ:+.2f}", "Gas-fraction engine", "EVo" if use_evo else "analytical fallback"]],
      left_cols=(0, 2))


In [ ]:
#@title 3) Magmatic levers: temperature, H2O/CO2/S budget and melt compressibility for each Vulcano composition  { display-mode: "form", run: "auto" }
#@markdown Defaults are the measured values compiled in Chapter 4 (Scenario A).
#@markdown Temperatures: Clocchiatti et al. (1994), Gioncada et al. (1998), Fusillo et al. (2015).
#@markdown H2O: melt inclusions - 1.0-1.5 wt% for the 1888-90 suite, 0.8-1.9 wt% latite, 0.36-1.32 wt% Vulcanello.
#@markdown CO2: below detection (<50 ppm) in every Vulcano melt inclusion; 220 ppm for the mafic end-member is the only published quantitative value (Paonita et al. 2013).
#@markdown S: Gioncada et al. (1998) and Fusillo et al. (2015).
#@markdown beta_liquid = 1/K of the melt; typical range (0.5-2)e-10 Pa^-1 (Spera 2000; Rivalta & Segall 2008).

#@markdown --- **RHYOLITE** (2 km reservoir and the shallow 2021 source level)
RHYOLITE_T_C = 1000     #@param {type:"slider", min:850, max:1150, step:5}
RHYOLITE_H2O_wt = 1.25  #@param {type:"number"}
RHYOLITE_CO2_wt = 0.005 #@param {type:"number"}
RHYOLITE_S_wt = 0.02    #@param {type:"number"}
RHYOLITE_beta_liquid = 1.2e-10 #@param {type:"number"}

#@markdown --- **TRACHYTE** (5 km reservoir)
TRACHYTE_T_C = 1075     #@param {type:"slider", min:900, max:1150, step:5}
TRACHYTE_H2O_wt = 1.25  #@param {type:"number"}
TRACHYTE_CO2_wt = 0.005 #@param {type:"number"}
TRACHYTE_S_wt = 0.07    #@param {type:"number"}
TRACHYTE_beta_liquid = 1.1e-10 #@param {type:"number"}

#@markdown --- **LATITE** (12 km reservoir)
LATITE_T_C = 1080       #@param {type:"slider", min:950, max:1200, step:5}
LATITE_H2O_wt = 1.35    #@param {type:"number"}
LATITE_CO2_wt = 0.005   #@param {type:"number"}
LATITE_S_wt = 0.10      #@param {type:"number"}
LATITE_beta_liquid = 1.0e-10 #@param {type:"number"}

#@markdown --- **SHOSHONITE** (recharge into the 12 km reservoir)
SHOSHONITE_T_C = 1100    #@param {type:"slider", min:1000, max:1250, step:5}
SHOSHONITE_H2O_wt = 0.85 #@param {type:"number"}
SHOSHONITE_CO2_wt = 0.022 #@param {type:"number"}
SHOSHONITE_S_wt = 0.03   #@param {type:"number"}
SHOSHONITE_beta_liquid = 0.8e-10 #@param {type:"number"}

MAGMA_STATE = {
    "RHYOLITE":   dict(T_C=float(RHYOLITE_T_C), H2O_wt=float(RHYOLITE_H2O_wt),
                       CO2_wt=float(RHYOLITE_CO2_wt), S_wt=float(RHYOLITE_S_wt),
                       beta_liquid=float(RHYOLITE_beta_liquid), rho_melt=2300.),
    "TRACHYTE":   dict(T_C=float(TRACHYTE_T_C), H2O_wt=float(TRACHYTE_H2O_wt),
                       CO2_wt=float(TRACHYTE_CO2_wt), S_wt=float(TRACHYTE_S_wt),
                       beta_liquid=float(TRACHYTE_beta_liquid), rho_melt=2400.),
    "LATITE":     dict(T_C=float(LATITE_T_C), H2O_wt=float(LATITE_H2O_wt),
                       CO2_wt=float(LATITE_CO2_wt), S_wt=float(LATITE_S_wt),
                       beta_liquid=float(LATITE_beta_liquid), rho_melt=2450.),
    "SHOSHONITE": dict(T_C=float(SHOSHONITE_T_C), H2O_wt=float(SHOSHONITE_H2O_wt),
                       CO2_wt=float(SHOSHONITE_CO2_wt), S_wt=float(SHOSHONITE_S_wt),
                       beta_liquid=float(SHOSHONITE_beta_liquid), rho_melt=2550.),
}
section("Magmatic input parameters (Scenario A)")
_src = {"RHYOLITE":"Clocchiatti et al. (1994); Gioncada et al. (1998)",
        "TRACHYTE":"Clocchiatti et al. (1994); Gioncada et al. (1998)",
        "LATITE":"Gioncada et al. (1998); Paonita et al. (2013)",
        "SHOSHONITE":"Fusillo et al. (2015); Paonita et al. (2013)"}
table(["Magma", "T (°C)", "H<sub>2</sub>O (wt%)", "CO<sub>2</sub> (wt%)", "S (wt%)",
       "β<sub>liquid</sub> (Pa<sup>−1</sup>)", "Main sources"],
      [[k.capitalize(), f"{v['T_C']:.0f}", f"{v['H2O_wt']:.2f}", f"{v['CO2_wt']:.3f}",
        f"{v['S_wt']:.2f}", sci(v['beta_liquid']), _src[k]] for k, v in MAGMA_STATE.items()],
      left_cols=(0, 6),
      caption="Total (dissolved + exsolved) volatile budgets supplied to the EVo saturation search. "
              "CO<sub>2</sub> is below detection (&lt;50 ppm) in every Vulcano melt inclusion; 220 ppm is the only "
              "published quantitative value, for the mafic end-member (Paonita et al. 2013).")


In [ ]:
#@title 4) Physics engine: Vulcano melt compositions, EVo wrapper, compressibilities and deformation models  { display-mode: "form" }
import io, os, re, sys, contextlib, hashlib, json, tempfile
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# 1) REAL VULCANO (LA FOSSA) MELT COMPOSITIONS  [wt%, volatile-free]
#    Sources: De Astis et al. (1997, J.Pet; 2013 GSL Mem 37); Gioncada et al.
#    (1998, Bull.Volc.); Clocchiatti et al. (1994); Piochi et al. (2009);
#    Fusillo et al. (2015); Costa et al. (2020); Nicotra et al. (2018).
# ----------------------------------------------------------------------------
COMPOSITIONS = {
    # Gioncada et al. (1998) Bull.Volc. 60, Table 1 -- bulk-rock analyses of the
    # very samples used for the melt-inclusion work. All Fe as Fe2O3 in the
    # original; converted here to FeO(tot) = 0.8998 x Fe2O3(tot) for EVo.
    # GS91-50c, 1888-90 eruption, La Fossa (RY)
    "RHYOLITE": dict(SIO2=73.54, TIO2=0.13, AL2O3=13.10, FEO=2.21, MNO=0.07,
                     MGO=0.34, CAO=1.02, NA2O=4.35, K2O=4.95, P2O5=0.04),
    # Palizzi-pumices TRACHYTE, Gioncada et al. (1998) Table 1: the TR column
    # under "Palizzi pumices" is printed WITHOUT a sample ID in the original
    # table (verified from word coordinates in the PDF); GS93-71 is the
    # adjacent Palizzi LATITE (SiO2 57.36). SiO2 61.6 keeps this analysis
    # inside the EVo 'phonolite' solubility class (52-63 wt% SiO2).
    # Alternative WITH a printed ID (1888-90 suite): GS91-42a SiO2 65.43,
    # TiO2 0.38, Al2O3 14.71, FeO 4.45, MnO 0.11, MgO 1.72, CaO 3.25,
    # Na2O 3.99, K2O 5.27, P2O5 0.21 (falls in the EVo 'rhyolite' class)
    "TRACHYTE": dict(SIO2=61.60, TIO2=0.61, AL2O3=17.51, FEO=4.30, MNO=0.10,
                     MGO=1.25, CAO=2.25, NA2O=4.42, K2O=7.25, P2O5=0.22),
    # GS91-17, Pietre Cotte, La Fossa (LT)
    "LATITE":   dict(SIO2=57.45, TIO2=0.61, AL2O3=16.83, FEO=6.60, MNO=0.14,
                     MGO=2.58, CAO=5.17, NA2O=3.81, K2O=5.67, P2O5=0.43),
    # GS91-66, Vulcanello I (SH)
    "SHOSHONITE": dict(SIO2=53.79, TIO2=0.70, AL2O3=15.08, FEO=8.16, MNO=0.16,
                       MGO=4.70, CAO=7.67, NA2O=3.55, K2O=4.89, P2O5=0.39),
}
# EVo solubility class (Burgisser et al. 2015 fits) with SiO2 validity ranges:
# basalt 45-55, phonolite 52-63, rhyolite 65-80 wt% SiO2
EVO_CLASS = {"RHYOLITE": "rhyolite", "TRACHYTE": "phonolite",
             "LATITE": "phonolite", "SHOSHONITE": "basalt"}

# ----------------------------------------------------------------------------
# 2) MAGMATIC STATE PARAMETERS (the "levers")
#    T:   Clocchiatti et al. 1994 (rhyolite ~1000 C, latite/trachyte 1050-1100),
#         Gioncada et al. 1998 (primitive basalt 1180+-20 C)
#    H2O: Rossi/Mollo hygrometry (trachyte ~3-3.5, latite ~2-2.5 wt%),
#         Pal B rhyolite 1.7-2.2 wt%; MIs 1.0-2.5 wt% (Clocchiatti et al. 1994)
#    CO2: MIs are CO2-poor (degassed before entrapment; Gioncada et al. 1998)
#         but gas chemistry requires a deep CO2-rich budget
#         (Paonita et al. 2013; Mandarano et al. 2016) -> totals of 0.1-0.4 wt%
#    S:   Gioncada et al. 1998 (S-rich primitive melts ~0.2-0.3 wt%)
#    beta_liquid: melt compressibility 1/K_melt, typical (0.5-2)e-10 1/Pa
#         (Spera 2000; Rivalta & Segall 2008; Kilbride et al. 2016)
# ----------------------------------------------------------------------------
# MAGMA_STATE  <- set by the lever cell above

# PHYS  <- set by the lever cell above

R_GAS = 8.314  # J/mol/K

# ----------------------------------------------------------------------------
# 3) PRESSURE FROM DEPTH  (step 1 of the workflow)
# ----------------------------------------------------------------------------
def pressure_Pa(depth_km, rho_crust=None, g=None):
    rho = PHYS["rho_crust"] if rho_crust is None else rho_crust
    gg = PHYS["g"] if g is None else g
    return rho * gg * depth_km * 1e3

# ----------------------------------------------------------------------------
# 4) EVo WRAPPER  (step 2: phi + fugacities at chamber P,T)
#    Runs EVo (Liggins et al. 2020, 2022; github.com/pipliggins/EVo) in
#    FIND_SATURATION mode with the melt volatile budget, closed-system
#    decompression from P_sat down to the chamber pressure.
# ----------------------------------------------------------------------------
ENV_TEMPLATE = """COMPOSITION: {evoclass}
RUN_TYPE: closed
SINGLE_STEP: False
FIND_SATURATION: True
ATOMIC_MASS_SET: False
GAS_SYS: cohs
FE_SYSTEM: True
OCS: False
S_SAT_WARN: False
T_START: {T_K}
P_START: 3000
P_STOP: {P_stop_bar}
DP_MIN: 0.1
DP_MAX: 50
MASS: 100
WgT: 0.00001
LOSS_FRAC: 0.9999
DENSITY_MODEL: spera2000
FO2_MODEL: kc1991
FMQ_MODEL: frost1991
H2O_MODEL: burguisser2015
H2_MODEL: gaillard2003
C_MODEL: burguisser2015
CO_MODEL: None
CH4_MODEL: None
SULFIDE_CAPACITY: oneill2020
SULFATE_CAPACITY: nash2019
SCSS: liu2007
N_MODEL: libourel2003
FO2_buffer_SET: True
FO2_buffer: FMQ
FO2_buffer_START: {dFMQ}
FO2_SET: False
FO2_START: 8.3e-12
FH2_SET: False
FH2_START: 0.24
FH2O_SET: False
FH2O_START: 1000
FCO2_SET: False
FCO2_START: 1
WTH2O_SET: True
WTH2O_START: {H2O_frac}
WTCO2_SET: True
WTCO2_START: {CO2_frac}
SULFUR_SET: True
SULFUR_START: {S_frac}
NITROGEN_SET: False
NITROGEN_START: 0.0001
GRAPHITE_SATURATED: False
GRAPHITE_START: 0.0001
"""

_EVO_CACHE = {}

def _write_chem(comp_key, path):
    with open(path, "w") as f:
        for ox, val in COMPOSITIONS[comp_key].items():
            f.write(f"{ox}: {val}\n")

def _evo_exit(msg=None):
    raise SystemExit(msg)

_GAS_SPECIES = ("mH2O", "mH2", "mCO2", "mCO", "mCH4", "mSO2", "mH2S", "mS2", "mO2")

def _evo_extras(last):
    """Full equilibrium vapour speciation (mole fractions) and residual melt volatiles."""
    sp = {k[1:]: float(last[k]) for k in _GAS_SPECIES if k in last.index}
    out = dict(gas_species=sp)
    for key, col in (("melt_H2O_wt", "H2O_melt"), ("melt_CO2_wt", "CO2_melt"),
                     ("melt_S_wt", "Stot_melt")):
        if col in last.index:
            out[key] = float(last[col])
    return out

def run_evo_state(comp_key, P_MPa, state=None, dFMQ=None,
                  workdir=os.path.join(tempfile.gettempdir(), "lafossa_evo")):
    """Return the magmatic state at (comp, T, P): phi, fugacities, densities."""
    st = MAGMA_STATE[comp_key] if state is None else state
    dfmq = PHYS["dFMQ"] if dFMQ is None else dFMQ
    key = (comp_key, round(P_MPa, 3), st["T_C"], st["H2O_wt"], st["CO2_wt"],
           st["S_wt"], dfmq)
    if key in _EVO_CACHE:
        return _EVO_CACHE[key]

    os.makedirs(workdir, exist_ok=True)
    tag = hashlib.md5(json.dumps(key).encode()).hexdigest()[:8]
    chem = os.path.join(workdir, f"chem_{tag}.yaml")
    env = os.path.join(workdir, f"env_{tag}.yaml")
    _write_chem(comp_key, chem)
    with open(env, "w") as f:
        f.write(ENV_TEMPLATE.format(
            evoclass=EVO_CLASS[comp_key], T_K=st["T_C"] + 273.15,
            P_stop_bar=max(int(round(P_MPa * 10.0)), 1), dFMQ=dfmq,
            # floors: the COHS system needs every species > 0; a slider set to
            # exactly 0 is treated as a trace amount (1 ppm)
            H2O_frac=max(st["H2O_wt"], 1e-3) / 100.0,
            CO2_frac=max(st["CO2_wt"], 1e-4) / 100.0,
            S_frac=max(st["S_wt"], 1e-4) / 100.0))

    res = dict(comp=comp_key, P_MPa=P_MPa, T_C=st["T_C"], phi=0.0,
               saturated=False, P_sat_MPa=np.nan, gas_molmass=np.nan,
               rho_melt=st["rho_melt"], rho_gas=np.nan,
               fO2_dFMQ=np.nan, fH2O=np.nan, fCO2=np.nan, fSO2=np.nan,
               fH2S=np.nan, fS2=np.nan, gas_wt=0.0,
               XH2O_gas=np.nan, XCO2_gas=np.nan,
               beta_m_evo=np.nan, engine="EVo", gas_species={})
    buf = io.StringIO()
    try:
        import evo
        from evo import messages as _msgs
        _msgs.query_yes_no = lambda *a, **k: True   # never block on prompts
        for _n, _m in list(sys.modules.items()):        # EVo's bare exit() must raise SystemExit in a kernel
            if _n == "evo" or _n.startswith("evo."):
                setattr(_m, "exit", _evo_exit)
        with contextlib.redirect_stdout(buf):
            df = evo.run_evo(chem, env, None, folder=os.path.join(workdir, tag))
        last = df.iloc[-1]
        prev = df.iloc[-2] if len(df) > 1 else last
        res.update(
            phi=float(last["Exsol_vol%"]) / 100.0,
            saturated=True,
            P_sat_MPa=float(df["P"].iloc[0]) / 10.0,
            gas_molmass=float(last["mol_mass"]),
            rho_melt=float(last["rho_melt"]),
            fO2_dFMQ=float(last["FMQ"]), fH2O=float(last["fH2O"]),
            fCO2=float(last["fCO2"]), fSO2=float(last["fSO2"]),
            fH2S=float(last["fH2S"]), fS2=float(last["fS2"]),
            gas_wt=float(last["Gas_wt"]) / 100.0,
            XH2O_gas=float(last["mH2O"]), XCO2_gas=float(last["mCO2"]),
        )
        res.update(_evo_extras(last))
        # gas density from bulk & melt densities and gas weight fraction
        wg = res["gas_wt"]
        if wg > 0:
            inv = 1.0 / float(last["rho_bulk"]) - (1 - wg) / res["rho_melt"]
            if inv > 0:
                res["rho_gas"] = wg / inv
        # effective full magma compressibility along the closed-system path
        # (diagnostic: includes exsolution; compare with phi-mixture value)
        if len(df) > 1 and last["P"] != prev["P"]:
            dP = (float(prev["P"]) - float(last["P"])) * 1e5   # Pa
            res["beta_m_evo"] = (np.log(float(prev["rho_bulk"]))
                                 - np.log(float(last["rho_bulk"]))) / dP
    except SystemExit as e:
        m = re.search(r"saturation pressure \(([\d.eE+-]+)\s*bar\)",
                      str(e) + " " + buf.getvalue())
        if m:
            res["P_sat_MPa"] = float(m.group(1)) / 10.0
        res["saturated"] = False        # undersaturated at chamber P: phi = 0
    except Exception as err:            # solver hiccup -> nudge P_STOP, retry
        done = False
        for bump in (1, 2, 5):
            try:
                with open(env) as f:
                    txt = f.read()
                p0 = int(round(max(P_MPa * 10.0, 1.0)))
                txt = re.sub(r"P_STOP: \d+", f"P_STOP: {p0 + bump}", txt)
                with open(env, "w") as f:
                    f.write(txt)
                with contextlib.redirect_stdout(buf):
                    df = evo.run_evo(chem, env, None,
                                     folder=os.path.join(workdir, tag))
                last = df.iloc[-1]
                prev = df.iloc[-2] if len(df) > 1 else last
                res.update(
                    phi=float(last["Exsol_vol%"]) / 100.0, saturated=True,
                    P_sat_MPa=float(df["P"].iloc[0]) / 10.0,
                    gas_molmass=float(last["mol_mass"]),
                    rho_melt=float(last["rho_melt"]),
                    fO2_dFMQ=float(last["FMQ"]), fH2O=float(last["fH2O"]),
                    fCO2=float(last["fCO2"]), fSO2=float(last["fSO2"]),
                    fH2S=float(last["fH2S"]), fS2=float(last["fS2"]),
                    gas_wt=float(last["Gas_wt"]) / 100.0,
                    XH2O_gas=float(last["mH2O"]), XCO2_gas=float(last["mCO2"]))
                res.update(_evo_extras(last))
                wg = res["gas_wt"]
                if wg > 0:
                    inv = 1.0 / float(last["rho_bulk"]) - (1 - wg) / res["rho_melt"]
                    if inv > 0:
                        res["rho_gas"] = wg / inv
                if len(df) > 1 and last["P"] != prev["P"]:
                    dP = (float(prev["P"]) - float(last["P"])) * 1e5
                    res["beta_m_evo"] = (np.log(float(prev["rho_bulk"]))
                                         - np.log(float(last["rho_bulk"]))) / dP
                res["engine"] = f"EVo (P_STOP+{bump} bar)"
                done = True
                break
            except SystemExit as e2:
                m = re.search(r"saturation pressure \(([\d.eE+-]+)\s*bar\)",
                              str(e2))
                if m:
                    res["P_sat_MPa"] = float(m.group(1)) / 10.0
                res["saturated"] = False
                done = True
                break
            except Exception:
                continue
        if not done:                     # EVo unavailable/crashed -> fallback
            res = simple_phi_model(comp_key, P_MPa, st)
            res["engine"] = f"fallback ({type(err).__name__})"
    _EVO_CACHE[key] = res
    return res

# ----------------------------------------------------------------------------
# 4b) ANALYTIC FALLBACK (only used if EVo cannot run)
#     sqrt-law H2O solubility + Henry CO2, ideal-gas mixture
# ----------------------------------------------------------------------------
SOL = {"RHYOLITE": (0.41, 3.8), "TRACHYTE": (0.39, 4.5),
       "LATITE": (0.39, 5.0), "SHOSHONITE": (0.36, 5.5)}  # (wt%/sqrt(MPa), ppm/MPa)

def simple_phi_model(comp_key, P_MPa, st):
    sw, kc = SOL[comp_key]
    T_K = st["T_C"] + 273.15
    h2o_sat = sw * np.sqrt(P_MPa)                 # wt%
    co2_sat = kc * P_MPa * 1e-4                   # wt%
    ex_h2o = max(st["H2O_wt"] - h2o_sat, 0.0) / 100.0
    ex_co2 = max(st["CO2_wt"] - co2_sat, 0.0) / 100.0
    wg = ex_h2o + ex_co2
    res = dict(comp=comp_key, P_MPa=P_MPa, T_C=st["T_C"], phi=0.0,
               saturated=wg > 0, P_sat_MPa=np.nan, gas_molmass=np.nan,
               rho_melt=st["rho_melt"], rho_gas=np.nan, fO2_dFMQ=np.nan,
               fH2O=np.nan, fCO2=np.nan, fSO2=np.nan, fH2S=np.nan, fS2=np.nan,
               gas_wt=wg, XH2O_gas=np.nan, XCO2_gas=np.nan,
               beta_m_evo=np.nan, engine="fallback")
    if wg <= 0:
        return res
    n_h2o, n_co2 = ex_h2o / 18.015e-3, ex_co2 / 44.01e-3
    M = (ex_h2o + ex_co2) / (n_h2o + n_co2)
    rho_g = P_MPa * 1e6 * M / (R_GAS * T_K)
    Vg, Vl = wg / rho_g, (1 - wg) / st["rho_melt"]
    res.update(phi=Vg / (Vg + Vl), gas_molmass=M, rho_gas=rho_g,
               XH2O_gas=n_h2o / (n_h2o + n_co2), XCO2_gas=n_co2 / (n_h2o + n_co2))
    return res

# ----------------------------------------------------------------------------
# 5) COMPRESSIBILITIES  (steps 2-5)
# ----------------------------------------------------------------------------
def beta_gas(P_Pa):
    """Ideal-gas isothermal compressibility, 1/P (Rivalta & Segall 2008)."""
    return 1.0 / P_Pa

def beta_magma(phi, beta_vol, beta_liq):
    """beta_m = phi*beta_volatile + (1-phi)*beta_liquid  (step 3)."""
    return phi * beta_vol + (1.0 - phi) * beta_liq

# --- deformation-model classes (dMODELS / Battaglia et al. 2013, python port)
import dmodelspy.sill as _sillmod

def _gauleg_fixed(x1, x2, N):
    z, w = np.polynomial.legendre.leggauss(int(N))
    return 0.5 * (x2 + x1) + 0.5 * (x2 - x1) * z, 0.5 * (x2 - x1) * w
_sillmod._gauleg = _gauleg_fixed          # numpy>=2 compatibility patch

from dmodelspy import Sill as _Sill, Spheroid as _Spheroid

class SillFixed(_Sill):
    """Fialko et al. (2001) penny-shaped crack, numpy-2-safe displacement."""
    def calc_displ(self, x, y, z):
        from numpy import array, sqrt, zeros, arange, sinh, cosh
        from scipy.special import jv as besselj
        if self._dV is None:
            self._calc_base()
        x = (array(x, ndmin=1, dtype=float) - self.x0) / self.a
        y = (array(y, ndmin=1, dtype=float) - self.y0) / self.a
        z = (array(z, ndmin=1, dtype=float) - self.z0) / self.a
        r = sqrt(x ** 2 + y ** 2)
        h = self.z0 / self.a
        Uz, Ur = zeros(r.shape), zeros(r.shape)
        for i in arange(r.size):
            czh = (z[i] + h) * self._csi_col
            J0 = besselj(0, r[i] * self._csi_col)
            Uzi = J0 * (((1 - 2 * self.nu) * self._Barr - czh * self._Aarr) * sinh(czh)
                        + (2 * (1 - self.nu) * self._Aarr - czh * self._Barr) * cosh(czh))
            Uz[i] = (self._wcsi_row @ Uzi).item()
            J1 = besselj(1, r[i] * self._csi_col)
            Uri = J1 * (((1 - 2 * self.nu) * self._Aarr + czh * self._Barr) * sinh(czh)
                        + (2 * (1 - self.nu) * self._Barr + czh * self._Aarr) * cosh(czh))
            Ur[i] = (self._wcsi_row @ Uri).item()
        rs = np.where(r == 0, 1.0, r)
        return (self.a * self.P_G * Ur * x / rs,
                self.a * self.P_G * Ur * y / rs,
                -self.a * self.P_G * Uz)

def mogi_uz(dVc, z0, r, nu):
    """Mogi (1958) point source, volume form: uz=(1-nu)dV z0 / (pi R^3)."""
    r = np.atleast_1d(np.asarray(r, dtype=float))
    return (1 - nu) * dVc / np.pi * z0 / (z0 ** 2 + r ** 2) ** 1.5

def beta_c_sphere(mu):
    """Chamber compressibility of a sphere: 3/(4 mu) (Segall 2010, eq. 8.25)."""
    return 3.0 / (4.0 * mu)

def spheroid_dV(a, aspect, dP, mu):
    """True cavity volume change of a pressurized prolate spheroid
    (full-space Eshelby expression, Amoruso & Crescentini 2009, as adopted
    for the Yang model by Battaglia et al. 2013, dMODELS):
        dV = pi a b^2 (dP/mu) [A^2/3 - 0.7A + 1.37],  A = b/a (0 < A <= 1)
    """
    A = aspect
    return np.pi * a * (A * a) ** 2 * dP / mu * (A ** 2 / 3.0 - 0.7 * A + 1.37)

def yang_source(V0, aspect, z0, mu, nu, dip, strike, dP_ref=None):
    """Build the Yang (1988)/Newman (2006) spheroid with volume V0 (m3)."""
    a = (3.0 * V0 / (4.0 * np.pi * aspect ** 2)) ** (1.0 / 3.0)
    dP = 1e-4 * mu if dP_ref is None else dP_ref
    sp = _Spheroid(0., 0., z0, a=a, asrat=aspect, P_G=dP / mu, mu=mu, nu=nu,
                   theta=min(dip, 89.99), phi=strike)
    return sp, a, dP

def beta_c_yang(V0, aspect, z0, mu, nu, dip, strike):
    """beta_c = (3/4mu) [A^2/3 - 0.7A + 1.37]  (sphere: 3/4mu at A=1)."""
    a = (3.0 * V0 / (4.0 * np.pi * aspect ** 2)) ** (1.0 / 3.0)
    return spheroid_dV(a, aspect, 1.0, mu) / V0

def penny_source(a, z0, mu, nu, dP_ref=None):
    dP = 1e-4 * mu if dP_ref is None else dP_ref
    return SillFixed(0., 0., z0, P_G=dP / mu, a=a, nu=nu), dP

def beta_c_penny(a, V0, z0, mu, nu):
    """dV/dP of a Fialko half-space crack, normalised by reservoir volume V0."""
    s, dP = penny_source(a, z0, mu, nu)
    return s.dV / (V0 * dP)

# ----------------------------------------------------------------------------
# 6-7) rV FACTOR, CHAMBER VOLUME CHANGE, UPLIFT
# ----------------------------------------------------------------------------
def rV_factor(beta_m, beta_c):
    return 1.0 + beta_m / beta_c

def uplift_mogi(dVc, z0, dists, nu):
    return mogi_uz(dVc, z0, np.asarray(dists), nu)

def uplift_yang(dVc, V0, aspect, z0, mu, nu, dip, strike, dists, azimuth=0.0):
    sp, a, dP = yang_source(V0, aspect, z0, mu, nu, dip, strike)
    scale = dVc / spheroid_dV(a, aspect, dP, mu)   # true cavity dV (A&C 2009)
    az = np.deg2rad(azimuth)
    d = np.asarray(dists, dtype=float)
    x, y = d * np.sin(az), d * np.cos(az)     # az measured from North
    u, v, w = sp.calc_displ(x, y, np.zeros_like(d))
    return w * scale

def uplift_penny(dVc, a, z0, mu, nu, dists):
    s, dP = penny_source(a, z0, mu, nu)
    scale = dVc / s.dV
    d = np.asarray(dists, dtype=float)
    u, v, w = s.calc_displ(d, np.zeros_like(d), np.zeros_like(d))
    return w * scale

section("Physics engine ready — melt compositions used by EVo")
_ox = ["SIO2","TIO2","AL2O3","FEO","MNO","MGO","CAO","NA2O","K2O","P2O5"]
_lab = ["SiO<sub>2</sub>","TiO<sub>2</sub>","Al<sub>2</sub>O<sub>3</sub>","FeO<sub>tot</sub>","MnO",
        "MgO","CaO","Na<sub>2</sub>O","K<sub>2</sub>O","P<sub>2</sub>O<sub>5</sub>"]
_samp = {"RHYOLITE":"GS91-50c (1888–90)", "TRACHYTE":"Palizzi pumices, TR column",
         "LATITE":"GS91-17 (Pietre Cotte)", "SHOSHONITE":"GS91-66 (Vulcanello I)"}
table(["Oxide (wt%)"] + [k.capitalize() for k in COMPOSITIONS],
      [["Sample"] + [_samp[k] for k in COMPOSITIONS]] +
      [[l] + [f"{COMPOSITIONS[k][o]:.2f}" for k in COMPOSITIONS] for o, l in zip(_ox, _lab)] +
      [["EVo solubility class"] + [EVO_CLASS[k] for k in COMPOSITIONS]],
      caption="Gioncada et al. (1998), Table 1 (all Fe converted to FeO<sub>tot</sub> = 0.8998 × Fe<sub>2</sub>O<sub>3</sub>). "
              "The EVo solubility class is the Burgisser et al. (2015) fit whose SiO<sub>2</sub> range contains each melt.")


In [ ]:
#@title 5) Run the seven-step chain — magmatic state, compressibilities, r_V and volume changes  { display-mode: "form" }

DISTS_M = [0.0, 1000.0, 2000.0]          # uplift at centre, 1 km, 2 km

# ------------------------------------------------------------------ scenarios
MOGI_ROWS = [
    dict(depth_km=2.0,  chamber="RHYOLITE", intrusion="TRACHYTE",
         V0=5e8,  Ve=[1e6, 5e6, 1e7]),
    dict(depth_km=5.0,  chamber="TRACHYTE", intrusion="LATITE",
         V0=5e9,  Ve=[1e6, 5e6, 1e7]),
    dict(depth_km=12.0, chamber="LATITE",   intrusion="SHOSHONITE",
         V0=5e10, Ve=[1e6, 5e6, 1e7]),
]
YANG_ROWS = [
    # Di Traglia et al. (2023) Table S1, nu=0.35: depth 598 m, V0 = 8.83e6 m3
    # (the scheme's 0.5 km / 5e7 m3 are kept as a commented alternative)
    dict(depth_km=0.598, chamber="RHYOLITE", intrusion="TRACHYTE",
         V0=8.83e6, Ve=[1e5, 5e5, 1e6]),
]
PENNY_RADII_KM = [0.5, 1.0, 2.0]

# ------------------------------------------------------------------ one row
def compute_row(model, row, penny_a_km=None, P=None):
    """Apply the 7-step workflow to one table row; return a result dict."""
    p = {**PHYS, **(P or {})}
    d_m = row["depth_km"] * 1e3
    z_P = (p.get("yang_overburden_km") or row["depth_km"]) if model == "YANG" \
        else row["depth_km"]                    # pressure datum (see PHYS note)
    P_Pa = pressure_Pa(z_P, p["rho_crust"], p["g"])                   # (1)
    if p["use_evo"]:                                                  # (2)
        st = run_evo_state(row["chamber"], P_Pa / 1e6)
    else:
        st = simple_phi_model(row["chamber"], P_Pa / 1e6,
                                 MAGMA_STATE[row["chamber"]])
    phi = st["phi"]
    b_liq = MAGMA_STATE[row["chamber"]]["beta_liquid"]
    b_gas = beta_gas(P_Pa)
    b_m = beta_magma(phi, b_gas, b_liq)                               # (3)

    if model == "MOGI":                                                  # (4)
        mu, nu = p["mu_crust_GPa"] * 1e9, p["nu"]
        b_c = beta_c_sphere(mu)
    elif model == "YANG":
        mu, nu = p["mu_shallow_GPa"] * 1e9, p["nu_shallow"]
        b_c = beta_c_yang(row["V0"], p["yang_aspect"], d_m, mu, nu,
                             p["yang_dip"], p["yang_strike"])
    elif model == "PENNY":
        mu, nu = p["mu_crust_GPa"] * 1e9, p["nu"]
        b_c = beta_c_penny(penny_a_km * 1e3, row["V0"], d_m, mu, nu)

    rV = rV_factor(b_m, b_c)                                          # (5)
    out = dict(model=model, penny_radius_km=penny_a_km,
               depth_km=row["depth_km"], chamber=row["chamber"],
               intrusion=row["intrusion"], V0_m3=row["V0"],
               P_MPa=P_Pa / 1e6, T_C=st["T_C"], engine=st["engine"],
               saturated=st["saturated"], P_sat_MPa=st["P_sat_MPa"],
               phi=phi, XH2O_gas=st["XH2O_gas"], XCO2_gas=st["XCO2_gas"],
               fO2_dFMQ=st["fO2_dFMQ"], fH2O_bar=st["fH2O"],
               fCO2_bar=st["fCO2"], fSO2_bar=st["fSO2"],
               rho_melt=st["rho_melt"],
               beta_liquid=b_liq, beta_gas=b_gas, beta_m=b_m,
               beta_m_evo_effective=st["beta_m_evo"],
               beta_c=b_c, rV=rV, mu_Pa=mu, nu=nu)

    for k, Ve in enumerate(row["Ve"], start=1):
        dVc = Ve / rV                                                    # (6)
        if model == "MOGI":                                              # (7)
            uz = uplift_mogi(dVc, d_m, DISTS_M, nu)
        elif model == "YANG":
            uz = uplift_yang(dVc, row["V0"], p["yang_aspect"], d_m, mu,
                                nu, p["yang_dip"], p["yang_strike"],
                                DISTS_M, p["profile_azimuth"])
        elif model == "PENNY":
            uz = uplift_penny(dVc, penny_a_km * 1e3, d_m, mu, nu, DISTS_M)
        out[f"Ve{k}_m3"] = Ve
        out[f"dVc{k}_m3"] = dVc
        for dist, w in zip(DISTS_M, uz):
            out[f"uz{k}_{int(dist/1000)}km_mm"] = w * 1e3
    return out

def compute_all(P=None):
    rows = []
    for r in MOGI_ROWS:
        rows.append(compute_row("MOGI", r, P=P))
    for r in YANG_ROWS:
        rows.append(compute_row("YANG", r, P=P))
    for a in PENNY_RADII_KM:
        for r in MOGI_ROWS:                     # same rows, crack geometry
            rows.append(compute_row("PENNY", r, penny_a_km=a, P=P))
    return pd.DataFrame(rows)



res = compute_all()

# =========================================================== STEPS 1-2: magmatic state
section("Steps 1–2 · Pressure and magmatic state of every reservoir (EVo)")

def _state(comp, P_MPa):
    if PHYS["use_evo"]:
        return run_evo_state(comp, P_MPa)
    return simple_phi_model(comp, P_MPa, MAGMA_STATE[comp])

_levels = [("2021 source level", PHYS["yang_overburden_km"], r["chamber"], r["intrusion"]) for r in YANG_ROWS] + \
          [(f"{r['depth_km']:g} km reservoir", r["depth_km"], r["chamber"], r["intrusion"]) for r in MOGI_ROWS]
_rows, _sat = [], []
for lab, z, res_c, inj_c in _levels:
    P = pressure_Pa(z, PHYS["rho_crust"], PHYS["g"]) / 1e6
    for role, comp in (("resident", res_c), ("injected (reference)", inj_c)):
        st = _state(comp, P)
        _rows.append([lab, f"{z:g}", f"{P:.2f}", role, comp.capitalize(),
                      f"{MAGMA_STATE[comp]['T_C']:.0f}", num(st["P_sat_MPa"], 1),
                      "yes" if st["saturated"] else "no", num(100 * st["phi"], 2),
                      num(100 * st.get("gas_wt", 0.0), 3)])
        if role == "resident" and st["saturated"]:
            _sat.append((lab, comp, P, st))
table(["Level", "Depth (km)", "P (MPa)", "Role", "Magma", "T (°C)", "P<sub>sat</sub> (MPa)",
       "Saturated?", "φ (vol%)", "Gas (wt%)"], _rows,
      classes=lambda i, j: ("good" if _rows[i][7] == "yes" else None) if j in (7, 8) else None,
      caption="P = ρ<sub>c</sub> g z (Eq. 3.1). P<sub>sat</sub> is the pressure at which the magma's total volatile "
              "budget just saturates; below it a vapour phase exists. β<sub>m</sub> is computed from the <b>resident</b> "
              "magma; the injected magma is listed for reference only.")

for lab, comp, P, st in _sat:
    sp = st.get("gas_species", {}) or {}
    order = [("H2O", "H<sub>2</sub>O", "fH2O"), ("H2S", "H<sub>2</sub>S", "fH2S"),
             ("SO2", "SO<sub>2</sub>", "fSO2"), ("CO2", "CO<sub>2</sub>", "fCO2"),
             ("H2", "H<sub>2</sub>", None), ("S2", "S<sub>2</sub>", "fS2"),
             ("CO", "CO", None), ("CH4", "CH<sub>4</sub>", None)]
    rows = [[name, num(100 * sp[k], 3) if k in sp else "—",
             num(st.get(f), 3) if f and np.isfinite(st.get(f, np.nan)) else "—"]
            for k, name, f in order if k in sp or f]
    table(["Gas species", "mol%", "Fugacity (bar)"], rows,
          title=f"Equilibrium vapour of the saturated {comp.lower()} at {lab} ({P:.2f} MPa)",
          caption=(f"Mean molar mass {1000 * st['gas_molmass']:.1f} g mol<sup>−1</sup>; "
                   f"ρ<sub>gas</sub> = {num(st['rho_gas'], 1)} kg m<sup>−3</sup>; ρ<sub>melt</sub> = {num(st['rho_melt'], 0)} kg m<sup>−3</sup>; "
                   f"log fO<sub>2</sub> = FMQ {st['fO2_dFMQ']:+.2f}. Residual dissolved: "
                   f"H<sub>2</sub>O {num(st.get('melt_H2O_wt', np.nan), 3)} wt%, "
                   f"CO<sub>2</sub> {num(1e4 * st.get('melt_CO2_wt', np.nan), 1)} ppm, "
                   f"S {num(1e4 * st.get('melt_S_wt', np.nan), 0)} ppm."))

_rh = run_evo_state("RHYOLITE", 49.0) if PHYS["use_evo"] else None
if _rh is not None and np.isfinite(_rh["P_sat_MPa"]):
    note(f"<b>Independent check.</b> The rhyolite with {MAGMA_STATE['RHYOLITE']['H2O_wt']:.2f} wt% H<sub>2</sub>O "
         f"saturates at <b>{_rh['P_sat_MPa']:.1f} MPa</b>. Fluid-inclusion barometry on 1888–90 xenoliths gives "
         "30–60 MPa (Clocchiatti et al. 1994) and gas-chemistry models adopt 38 MPa (Paonita et al. 2013): three "
         "independent estimates of the shallow storage pressure.", "ok")
_und = [lab for lab, z, c, _ in _levels
        if not _state(c, pressure_Pa(z, PHYS["rho_crust"], PHYS["g"]) / 1e6)["saturated"]]
if _und:
    note("Undersaturated resident magmas (φ = 0, so β<sub>m</sub> = β<sub>liquid</sub>): " + ", ".join(_und) +
         ". The melt-inclusion budgets were trapped at shallow level after degassing, so they cannot saturate "
         "the magmas at their storage pressures — they constrain the shallow system only.")

# =========================================================== STEPS 3-5
section("Steps 3–5 · Magma compressibility, chamber compressibility and r<sub>V</sub>")
rows = []
for _, r in res.iterrows():
    rows.append([cfg_name(r), f"{r.depth_km:g}", f"{r.P_MPa:.2f}", num(100 * r.phi, 2),
                 sci(r.beta_liquid), sci(r.beta_gas), sci(r.beta_m), sci(r.beta_c),
                 f"{r.beta_c * r.mu_Pa:.3f}", f"{r.rV:.3f}", f"{100 / r.rV:.1f}%"])
table(["Configuration", "z (km)", "P (MPa)", "φ (vol%)", "β<sub>liquid</sub>", "β<sub>gas</sub> = 1/P",
       "β<sub>m</sub>", "β<sub>c</sub>", "β<sub>c</sub>·μ", "r<sub>V</sub>", "Expressed 1/r<sub>V</sub>"], rows,
      classes=lambda i, j: "hi" if (j in (6, 9) and res.iloc[i].phi > 0) else None,
      caption="β<sub>m</sub> = φβ<sub>gas</sub> + (1−φ)β<sub>liquid</sub> (Eq. 3.12); β<sub>c</sub> from the reservoir shape "
              "(sphere 3/4μ; prolate spheroid, Amoruso & Crescentini 2009; penny crack, Fialko et al. 2001, half-space); "
              "r<sub>V</sub> = 1 + β<sub>m</sub>/β<sub>c</sub> (Eq. 3.22). All compressibilities in Pa<sup>−1</sup>. "
              "Highlighted: configurations whose magma is gas-bearing.")
_eff = res[res.phi > 0]
for _, r in _eff.iterrows():
    if np.isfinite(r.beta_m_evo_effective):
        rv_eq = 1 + r.beta_m_evo_effective / r.beta_c
        note(f"<b>Frozen versus equilibrium compressibility ({cfg_name(r)}).</b> The frozen-phase mixture gives "
             f"β<sub>m</sub> = {sci(r.beta_m)} Pa<sup>−1</sup>; differencing the bulk density along the EVo equilibrium path "
             f"gives β<sub>m</sub><sup>eff</sup> = {sci(r.beta_m_evo_effective)} Pa<sup>−1</sup> "
             f"({r.beta_m_evo_effective / r.beta_m:.1f}×). With it r<sub>V</sub> would be {rv_eq:.1f} instead of "
             f"{r.rV:.1f}: an upper bound for slow pressurisation.")

# =========================================================== STEP 6
section("Step 6 · Reservoir volume change ΔV<sub>c</sub> = V<sub>e</sub> / r<sub>V</sub>")
rows = []
for _, r in res.iterrows():
    rows.append([cfg_name(r), f"{r.depth_km:g}", f"{r.rV:.3f}"] +
                [f"{sci(r[f'Ve{k}_m3'], 1)} → {sci(r[f'dVc{k}_m3'], 2)}" for k in (1, 2, 3)] +
                [f"{100 / r.rV:.1f}%"])
table(["Configuration", "z (km)", "r<sub>V</sub>", "V<sub>e1</sub> → ΔV<sub>c1</sub> (m³)",
       "V<sub>e2</sub> → ΔV<sub>c2</sub> (m³)", "V<sub>e3</sub> → ΔV<sub>c3</sub> (m³)", "Fraction expressed"], rows,
      caption="Each cell shows the injected volume and the resulting cavity volume change. "
              "The observed 2021 volume change was ≈7.3×10<sup>4</sup> m³ (Di Traglia et al. 2023, Table S1).")


In [ ]:
#@title 6) Step 7 · Predicted surface uplift — the completed FREYA scheme tables  { display-mode: "form" }
#@markdown The five scheme tables (Mogi, Yang, and penny-shaped cracks of 0.5, 1 and 2 km radius) are shown
#@markdown below exactly in the layout of `final_scheme_FREYA.docx`, with every UPLIFT cell filled in.

def _sci_plain(v):
    e = int(np.floor(np.log10(v)))
    m = v / 10 ** e
    return (f"{m:g}×10<sup>{e}</sup>").replace("-", "−")

def _uplift_cell(r, dist):
    return "<br>".join(f"V{k}:&nbsp;{mm(r[f'uz{k}_{dist}_mm'])}".replace(" ", "&nbsp;") for k in (1, 2, 3))

_SCHEME = [("MOGI MODEL", res[res.model == "MOGI"]),
           ("YANG MODEL (deformation source of Di Traglia et al. 2023)", res[res.model == "YANG"])]
for _a in PENNY_RADII_KM:
    _SCHEME.append((f"PENNY-SHAPED MODEL (radius: {_a:g} km)",
                    res[(res.model == "PENNY") & (res.penny_radius_km == _a)]))

section("Step 7 · Predicted surface uplift — completed scheme (final_scheme_FREYA)")
note("In each UPLIFT cell, <b>V1 / V2 / V3</b> are the uplifts produced by ADDED VOLUME 1 / 2 / 3 of the same row. "
     "Uplift is vertical displacement at the surface, above the source centre and at 1 and 2 km radial distance.")

_HDR = ["DEPTH", "CHAMBER COMPOSITION", "INTRUSION COMPOSITION", "INITIAL VOLUME (m³)",
        "ADDED VOLUME 1 (m³)", "ADDED VOLUME 2 (m³)", "ADDED VOLUME 3 (m³)",
        "UPLIFT AT CENTRE", "UPLIFT AT 1 KM", "UPLIFT AT 2 KM"]
for _title, _sub in _SCHEME:
    _rows = []
    for _, r in _sub.iterrows():
        _rows.append([f"{r.depth_km:g} km", r.chamber, r.intrusion, _sci_plain(r.V0_m3),
                      _sci_plain(r.Ve1_m3), _sci_plain(r.Ve2_m3), _sci_plain(r.Ve3_m3),
                      _uplift_cell(r, "0km"), _uplift_cell(r, "1km"), _uplift_cell(r, "2km")])
    table(_HDR, _rows, title=_title, left_cols=(7, 8, 9),
          caption=("r<sub>V</sub> of the rows (top to bottom): "
                   + ", ".join(f"{v:.3g}" for v in _sub.rV) + "."))

# ------------------------------------------------------------------ provenance note
_p = PHYS
_phis = "; ".join(f"{r.chamber.lower()} at {r.depth_km:g} km: φ = {100 * r.phi:.2f}%"
                  for _, r in res.drop_duplicates(["chamber", "depth_km"]).iterrows())
note("<b>How these numbers were obtained.</b> "
     f"P = ρgz with ρ = {_p['rho_crust']:.0f} kg m<sup>−3</sup>; gas exsolution and fugacities from EVo "
     f"(Liggins et al.; closed C–O–H–S system, fO<sub>2</sub> = FMQ {_p['dFMQ']:+.1f}), with melt compositions from "
     "Gioncada et al. (1998, Table 1) and volatile budgets from Clocchiatti et al. (1994), Gioncada et al. (1998), "
     "Fusillo et al. (2015) and Paonita et al. (2013); β<sub>m</sub> = φβ<sub>gas</sub> + (1−φ)β<sub>liquid</sub> with "
     "β<sub>gas</sub> = 1/P; r<sub>V</sub> = 1 + β<sub>m</sub>/β<sub>c</sub>; ΔV<sub>c</sub> = V<sub>e</sub>/r<sub>V</sub>. "
     f"Sources: Mogi (1958) sphere (β<sub>c</sub> = 3/4μ, μ = {_p['mu_crust_GPa']:g} GPa, ν = {_p['nu']:.2f}); "
     "Yang et al. (1988) / Newman et al. (2006) prolate spheroid at the Di Traglia et al. (2023) source "
     f"(598 m b.s.l., aspect {_p['yang_aspect']:.3f}, dip {_p['yang_dip']:g}°, azimuth {_p['yang_strike']:g}°, "
     f"μ = {_p['mu_shallow_GPa']:g} GPa, ν = {_p['nu_shallow']:.2f}); Fialko et al. (2001) penny-shaped crack "
     "(dMODELS, Battaglia et al. 2013). Gas fractions at storage conditions — " + _phis + ".")

# ------------------------------------------------------------------ shape diagnostic
section("Shape of the uplift field — a depth indicator independent of volume and r<sub>V</sub>")
_rows = []
for _, r in res.iterrows():
    u0 = r["uz3_0km_mm"]
    _rows.append([cfg_name(r), f"{r.depth_km:g}", mm(u0),
                  f"{r['uz3_1km_mm'] / u0:.2f}", f"{r['uz3_2km_mm'] / u0:.2f}"])
table(["Configuration", "Depth (km)", "u<sub>z</sub>(0), largest V<sub>e</sub>",
       "u<sub>z</sub>(1 km) / u<sub>z</sub>(0)", "u<sub>z</sub>(2 km) / u<sub>z</sub>(0)"], _rows,
      caption="Because every source is linear in its strength, these ratios do not depend on the injected volume, "
              "on r<sub>V</sub> or on the shear modulus. A signal that has decayed to a few per cent of its peak within "
              "1–2 km points to a source shallower than ~1 km; one still at 80–95% at 2 km points to 5 km or deeper.")


In [ ]:
#@title 7) Detectability thresholds, mechanical admissibility, geometric checks and validation against 2021  { display-mode: "form" }
target_uplift_mm = 10.0   #@param {type:"number"}
max_admissible_dP_MPa = 10.0   #@param {type:"number"}

# ============================================================ inverse analysis
section(f"Inverse analysis · what is needed for {target_uplift_mm:g} mm of central uplift?")
inv = []
for _, r in res.iterrows():
    Ve = r.Ve3_m3 * target_uplift_mm / r.uz3_0km_mm        # linear chain
    dVc = Ve / r.rV
    dP = dVc / (r.V0_m3 * r.beta_c) / 1e6                   # Eq. 3.21
    inv.append(dict(cfg=cfg_name(r), depth=r.depth_km, rV=r.rV, Ve=Ve, dVc=dVc, dP=dP,
                    ok=dP <= max_admissible_dP_MPa))
inv = sorted(inv, key=lambda d: d["Ve"])
_rows = [[d["cfg"], f"{d['depth']:g}", f"{d['rV']:.2f}", sci(d["Ve"]), sci(d["dVc"]),
          f"{d['dP']:.2f}", "yes" if d["ok"] else "NO"] for d in inv]
table(["Configuration", "Depth (km)", "r<sub>V</sub>", "V<sub>e</sub> needed (m³)",
       "ΔV<sub>c</sub> (m³)", "ΔP needed (MPa)", "Admissible?"], _rows,
      classes=lambda i, j: (("good" if inv[i]["ok"] else "bad") if j in (5, 6) else None),
      caption=f"Ordered from the cheapest to the most expensive signal. ΔP = ΔV<sub>c</sub>/(β<sub>c</sub>V<sub>0</sub>) "
              f"(Eq. 3.21). A configuration is judged inadmissible when ΔP exceeds {max_admissible_dP_MPa:g} MPa, i.e. "
              "beyond both the 3–8 MPa inferred for the 2021 source and plausible host-rock strength: the walls would "
              "fail before such a signal could form.")
_bad = [d["cfg"] + f" at {d['depth']:g} km" for d in inv if not d["ok"]]
_deep = [d for d in inv if d["depth"] >= 12 and d["ok"]]
note(f"<b>Reading the table.</b> {len(inv) - len(_bad)} of {len(inv)} configurations reach "
     f"{target_uplift_mm:g} mm at an admissible overpressure." +
     (f" Inadmissible: {', '.join(_bad)}." if _bad else "") +
     (f" From 12 km depth at least {sci(min(d['Ve'] for d in _deep))} m³ of magma is needed: deep recharge "
      "below this volume is essentially invisible to geodesy." if _deep else ""),
     "warn" if _bad else "info")

# ============================================================ thin-crack consistency
section("Geometric consistency of the penny-shaped reservoirs")
_rows, _flags = [], []
for _, r in res[res.model == "PENNY"].iterrows():
    a = r.penny_radius_km * 1e3
    wbar = r.V0_m3 / (np.pi * a ** 2)
    ratio = wbar / (2 * a)
    _flags.append(ratio > 0.1)
    _rows.append([f"{r.penny_radius_km:g}", f"{r.depth_km:g}", sci(r.V0_m3, 0),
                  f"{wbar:,.0f}", f"{ratio:.3g}", "violated" if ratio > 0.1 else "satisfied",
                  f"{r.penny_radius_km / r.depth_km:.2f}"])
table(["a (km)", "Depth (km)", "V<sub>0</sub> (m³)", "Mean opening w̄ (m)", "w̄ / 2a", "Thin crack?", "a / d"],
      _rows, classes=lambda i, j: ("bad" if _flags[i] else "good") if j == 5 else
      ("hi" if (j == 6 and float(_rows[i][6]) >= 1.0) else None),
      caption="The crack solution assumes w̄/(2a) ≪ 1 (threshold 0.1 used here). Rows that violate it describe a "
              "reservoir of the given volume with the wall stiffness of a much smaller crack, not a physical sill; "
              "they produce the largest r<sub>V</sub> values and the inadmissible entries above. "
              "Highlighted a/d ≥ 1: the crack rim reaches the free surface, the limit of validity of the solution.")
note(f"{sum(_flags)} of {len(_flags)} penny-shaped configurations violate the thin-crack condition.",
     "warn" if any(_flags) else "ok")

# ============================================================ validation vs Di Traglia 2023
section("Validation against the published 2021 source (Di Traglia et al. 2023, Supporting Information)")
# published values, Table S1 (nu = 0.35) — used as fixed validation targets, independent of the levers
_A, _a_pub, _b_pub, _V0, _k = 0.099, 595.0, 59.0, 8.83e6, 11.6e-8
_mu, _nu, _dip, _az, _z = 1e9, 0.35, 69.0, 137.0, 598.0
_dV_pub, _dP_pub = 73108.0, 8.48e6

bcmu_mine = beta_c_yang(_V0, _A, _z, _mu, _nu, _dip, _az) * _mu
bcmu_pub = 1.0 / (_k * _V0)
dV_mine = bcmu_mine / _mu * _V0 * _dP_pub
dP_V0 = _dV_pub / (_V0 * bcmu_mine / _mu)
dP_axes = _dV_pub / spheroid_dV(_a_pub, _b_pub / _a_pub, 1.0, _mu)
uz_c = uplift_yang(_dV_pub, _V0, _A, _z, _mu, _nu, _dip, _az, [0.0, 1000.0, 2000.0])

# maximum uplift on a map grid and its position
_g = np.linspace(-800, 800, 161)   # 10 m spacing
_X, _Y = np.meshgrid(_g, _g)
_sp, _asp, _dPref = yang_source(_V0, _A, _z, _mu, _nu, _dip, _az)
_W = _sp.calc_displ(_X.ravel(), _Y.ravel(), np.zeros(_X.size))[2] * _dV_pub / spheroid_dV(_asp, _A, _dPref, _mu)
_imax = int(np.argmax(_W))
_xm, _ym = _X.ravel()[_imax], _Y.ravel()[_imax]
_az_max = (np.degrees(np.arctan2(_xm, _ym)) + 360) % 360

def _pct(a, b):
    return f"{100 * (a / b - 1):+.1f}%"

_rows = [
    ["β<sub>c</sub>·μ of the spheroid (Eq. 3.17)", f"{bcmu_mine:.3f}", f"{bcmu_pub:.3f} (from k = 11.6×10<sup>−8</sup> m<sup>−3</sup>)", _pct(bcmu_mine, bcmu_pub)],
    ["ΔV<sub>c</sub> at ΔP = 8.48 MPa", f"{dV_mine:,.0f} m³", "73,108 ± 6,900 m³", _pct(dV_mine, _dV_pub)],
    ["ΔP for ΔV<sub>c</sub> = 73,108 m³ (using V<sub>0</sub>)", f"{dP_V0 / 1e6:.2f} MPa", "8.48 MPa", _pct(dP_V0, _dP_pub)],
    ["ΔP for ΔV<sub>c</sub> = 73,108 m³ (using semi-axes 595/59 m)", f"{dP_axes / 1e6:.2f} MPa", "8.48 MPa", _pct(dP_axes, _dP_pub)],
    ["Vertical uplift above the centroid", mm(uz_c[0] * 1e3), "cm-scale (LOS, longer window)", "consistent"],
    ["Vertical uplift at 1 km / 2 km", f"{mm(uz_c[1] * 1e3)} / {mm(uz_c[2] * 1e3)}", "confined to the crater area", "consistent"],
    ["Maximum uplift and its position", f"{mm(_W[_imax] * 1e3)}, {np.hypot(_xm, _ym):.0f} m at azimuth {_az_max:.0f}°",
     "expected towards the up-dip tip (azimuth ≈ 317°)", "consistent"],
]
table(["Quantity", "This notebook", "Published", "Difference"], _rows,
      classes=lambda i, j: "good" if j == 3 else None,
      caption="Targets are fixed at the published Table S1 values (ν = 0.35, μ = 1 GPa), so this check does not depend "
              "on the levers above. Because Di Traglia et al. used dMODELS, which adopts the same Amoruso & Crescentini "
              "(2009) expression, the agreement demonstrates that the two implementations are equivalent; the physical "
              "accuracy of the expression itself is set by its exact sphere and needle limits (0.3% and 2.8%).")

# four-nu cross-check of Table S1
_S1 = {0.25: (855, 85, 2.62e7, 3.91e-8), 0.30: (703, 70, 1.46e7, 7.02e-8),
       0.35: (595, 59, 8.83e6, 11.6e-8), 0.40: (706, 71, 1.47e7, 6.94e-8), 0.45: (656, 66, 1.18e7, 8.66e-8)}
_rows = []
for _nu_i, (a_i, b_i, V_i, k_i) in _S1.items():
    m_i = beta_c_yang(V_i, b_i / a_i, 598.0, 1.0, _nu_i, _dip, _az)
    p_i = 1.0 / (k_i * V_i)
    _rows.append([f"{_nu_i:.2f}", f"{a_i} / {b_i}", f"{b_i / a_i:.4f}", sci(V_i), f"{m_i:.3f}", f"{p_i:.3f}",
                  _pct(m_i, p_i) + (" *" if _nu_i == 0.25 else "")])
table(["ν", "a / b (m)", "A", "V<sub>0</sub> (m³)", "β<sub>c</sub>μ this notebook", "β<sub>c</sub>μ from published k", "Difference"],
      _rows, caption="All five Poisson-ratio inversions of Table S1. * The k printed for ν = 0.25 (3.91×10<sup>−3</sup>) "
                     "is inconsistent with its neighbours by three orders of magnitude; it is read here as "
                     "3.91×10<sup>−8</sup> m<sup>−3</sup>, a probable misprint.")
note("<b>Validation passed.</b> The chain reproduces the published pressure–volume mechanics of the 2021 source "
     "to within a few tenths of a per cent, and the modelled uplift matches the amplitude and confinement of the "
     "observed deformation.", "ok")


In [ ]:
#@title 8) Figures and the complete numerical results  { display-mode: "form" }
#@markdown All figures are drawn directly on this page; nothing is saved to disk.

_r = np.linspace(0, 4000, 161)
_depths = sorted(res.depth_km.unique())
_pal = ["#2e8b57", "#c0392b", "#e08b1e", "#2c6fbb", "#7d3c98", "#7f8c8d"]
_col = {d: _pal[i % len(_pal)] for i, d in enumerate(_depths)}

def _profile(r, dists):
    d_m, dVc = r.depth_km * 1e3, r.Ve3_m3 / r.rV
    if r.model == "MOGI":
        return uplift_mogi(dVc, d_m, dists, r.nu)
    if r.model == "YANG":
        return uplift_yang(dVc, r.V0_m3, PHYS["yang_aspect"], d_m, r.mu_Pa, r.nu,
                           PHYS["yang_dip"], PHYS["yang_strike"], dists, PHYS["profile_azimuth"])
    return uplift_penny(dVc, r.penny_radius_km * 1e3, d_m, r.mu_Pa, r.nu, dists)

# ------------------------------------------------------------ Figure 1: radial profiles
section("Figure 1 · Radial profiles of vertical uplift for the largest injected volume")
fig, axes = plt.subplots(2, 3, figsize=(14, 7.6))
specs = [("MOGI", None, "(a) Mogi sphere"), ("YANG", None, "(b) Yang spheroid — 2021 source"),
         ("PENNY", 0.5, "(c) Penny crack, a = 0.5 km"), ("PENNY", 1.0, "(d) Penny crack, a = 1 km"),
         ("PENNY", 2.0, "(e) Penny crack, a = 2 km")]
for ax, (m, a, ttl) in zip(axes.flat, specs):
    sub = res[res.model == m] if a is None else res[(res.model == m) & (res.penny_radius_km == a)]
    for _, r in sub.iterrows():
        ax.plot(_r / 1e3, _profile(r, _r) * 1e3, color=_col[r.depth_km], lw=1.6,
                label=f"{r.depth_km:g} km, {r.chamber.capitalize()} (r$_V$ = {r.rV:.3g})")
    ax.set_title(ttl, fontsize=10); ax.grid(alpha=.25, lw=.5)
    ax.set_xlim(0, 4); ax.legend(fontsize=7.5, frameon=False)
    ax.set_xlabel("radial distance [km]"); ax.set_ylabel("vertical uplift [mm]")
# panel (f): normalised profiles = depth diagnostic
ax = axes.flat[5]
for _, r in res[res.model.isin(["MOGI", "YANG"])].iterrows():
    u = _profile(r, _r)
    ax.plot(_r / 1e3, u / u[0], color=_col[r.depth_km], lw=1.6, label=f"{r.depth_km:g} km ({cfg_name(r).split(' (')[0]})")
ax.set_title("(f) Normalised shape u$_z$(r)/u$_z$(0) — depth indicator", fontsize=10)
ax.set_xlim(0, 4); ax.set_ylim(0, 1.05); ax.grid(alpha=.25, lw=.5)
ax.legend(fontsize=7.5, frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.32))
ax.set_xlabel("radial distance [km]"); ax.set_ylabel("normalised uplift")
fig.tight_layout(); plt.show()

# ------------------------------------------------------------ Figure 2: r_V and detectability
section("Figure 2 · Volume-partitioning factor and detectability threshold")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.2))
labels = [f"{cfg_name(r).replace('Yang spheroid (2021 source)', 'Yang 2021').replace('Penny crack, ', 'Penny ')}\n{r.depth_km:g} km"
          for _, r in res.iterrows()]
a1.bar(range(len(res)), res.rV, color=[_col[d] for d in res.depth_km], edgecolor="k", lw=.4)
a1.set_yscale("log"); a1.axhline(1, color="k", ls=":", lw=.8)
a1.set_xticks(range(len(res))); a1.set_xticklabels(labels, rotation=90, fontsize=7)
a1.set_ylabel(r"$r_V = 1+\beta_m/\beta_c$"); a1.grid(axis="y", alpha=.25, lw=.5)
a1.set_title("(a) Volume-partitioning factor (dotted: full expression, r$_V$ = 1)", fontsize=10)
yy = np.arange(len(inv))
bars = a2.barh(yy, [d["Ve"] for d in inv], color=[_col[d["depth"]] for d in inv], edgecolor="k", lw=.4)
for b, d in zip(bars, inv):
    if not d["ok"]:
        b.set_hatch("///"); b.set_alpha(.55)
a2.set_yticks(yy); a2.set_yticklabels([f"{d['cfg'].replace('Yang spheroid (2021 source)', 'Yang 2021')} | {d['depth']:g} km"
                                       for d in inv], fontsize=7.5)
a2.set_xscale("log"); a2.grid(axis="x", alpha=.25, lw=.5)
a2.axvline(7.31e4, color="k", ls="--", lw=.9, label="observed 2021 ΔV$_c$ = 7.3×10$^4$ m$^3$")
a2.legend(fontsize=7.5, loc="lower right")
a2.set_xlabel(f"injected volume needed for {target_uplift_mm:g} mm central uplift [m$^3$]")
a2.set_title(f"(b) Detectability threshold (hatched: ΔP > {max_admissible_dP_MPa:g} MPa)", fontsize=10)
fig.tight_layout(); plt.show()

# ------------------------------------------------------------ Figure 3: map of the 2021 source
section("Figure 3 · Map view of the uplift produced by the 2021 source (observed ΔV<sub>c</sub>)")
fig, ax = plt.subplots(figsize=(6.4, 5.4))
cs = ax.contourf(_X, _Y, (_W * 1e3).reshape(_X.shape), levels=18, cmap="viridis")
ax.contour(_X, _Y, (_W * 1e3).reshape(_X.shape), levels=[5, 10, 20, 30], colors="w", linewidths=.6)
ax.plot(0, 0, "k+", ms=10, mew=1.5, label="centroid epicentre")
ax.plot(_xm, _ym, "r*", ms=11, label=f"maximum {_W[_imax] * 1e3:.1f} mm")
ax.set_aspect("equal"); ax.set_xlabel("east [m]"); ax.set_ylabel("north [m]")
ax.legend(fontsize=8, loc="lower right"); plt.colorbar(cs, ax=ax, label="vertical uplift [mm]")
ax.set_title("Yang spheroid, Di Traglia et al. (2023) geometry, ΔV$_c$ = 73,108 m$^3$", fontsize=9.5)
fig.tight_layout(); plt.show()
note(f"Uplift above the centroid: {mm(uz_c[0] * 1e3)}; maximum {mm(_W[_imax] * 1e3)} about "
     f"{np.hypot(_xm, _ym):.0f} m from the epicentre towards azimuth {_az_max:.0f}°, i.e. towards the shallow "
     "up-dip tip of the spheroid (trend + 180°). The cusp in the contours north-west of the maximum lies above that "
     "tip, only ~43 m below the model surface, where the analytical kernel is least accurate.")

# ------------------------------------------------------------ complete results
section("Complete numerical results (every quantity computed for every configuration)")
_full = res.copy()
_full.insert(0, "configuration", [cfg_name(r) for _, r in res.iterrows()])
_full = _full.drop(columns=["model", "penny_radius_km"])
_full["phi"] = 100 * _full["phi"]
_full = _full.rename(columns={"phi": "phi_vol%", "depth_km": "depth (km)", "P_MPa": "P (MPa)",
                              "P_sat_MPa": "P_sat (MPa)", "V0_m3": "V0 (m3)"})
pd.set_option("display.max_columns", 200); pd.set_option("display.width", 250)
try:
    _sty = (_full.style.format(precision=4, thousands=",")
            .format({c: "{:.3e}" for c in _full.columns if c.startswith("beta") or c.startswith("V") or c.startswith("dVc")})
            .set_properties(**{"font-size": "11px"})
            .set_table_styles([{"selector": "th", "props": [("font-size", "11px"), ("background", "#2E5D8A"),
                                                         ("color", "white")]}]))
    display(_sty)
except Exception:                       # pandas Styler needs jinja2; plain table otherwise
    display(_full.round(6))
note("Everything above was computed live in this session and displayed on the page; no Word, Excel or CSV file "
     "has been written. Change any lever in cells 2–3 or 7 and use <i>Runtime ▸ Run after</i> to recompute.", "ok")


### Data sources and methods
* **EVo** (fugacities and C-O-H-S gas speciation): Liggins, Shorttle & Rimmer (2020, EPSL); Liggins et al. (2022) — `github.com/pipliggins/EVo`
* **Melt compositions and temperatures:** Gioncada et al. (1998, Bull. Volcanol., Table 1); Clocchiatti et al. (1994, Bull. Volcanol.); Fusillo et al. (2015, Bull. Volcanol.)
* **Volatile budgets (melt-inclusion H2O and S, deep CO2):** Clocchiatti et al. (1994); Gioncada et al. (1998); Fusillo et al. (2015); Paonita et al. (2013, GCA)
* **2021 deformation source:** Di Traglia et al. (2023, GRL 50, e2023GL104952) — dipping prolate spheroid, 598 m b.s.l., aspect 0.099, dip 69°, azimuth 137°, ΔV = 7.31×10⁴ m³, ν = 0.35
* **Compressibility framework:** Rivalta & Segall (2008, GRL 35, L04306): $r_V = 1+\beta_m/\beta_c$; Kilbride et al. (2016, Nat. Commun. 7, 13744); Segall (2010)
* **Chamber compressibility by shape:** sphere $3/4\mu$ (Segall 2010); prolate spheroid: Amoruso & Crescentini (2009, JGR 114, B02210); penny crack: Sneddon (1946), Fialko et al. (2001) half-space
* **Deformation models:** Mogi (1958); Yang et al. (1988) with the corrections of Newman et al. (2006); Fialko et al. (2001) — dMODELS implementation (Battaglia et al. 2013; `dmodelspy`)

**Notes.** (1) In the scheme tables, each UPLIFT cell holds three values, V1/V2/V3, one per ADDED VOLUME column. (2) For the Mogi sphere $\beta_c=3/4\mu$ is independent of the initial volume, so $V_0$ enters only through the Yang and penny-crack models. (3) $\beta_{gas}=1/P$ (ideal gas); the column `beta_m_evo_effective` of the complete results gives the effective compressibility including equilibrium exsolution, an upper bound for slow pressurisation. (4) If a magma is undersaturated at reservoir pressure, $\varphi=0$ and $P_{sat}$ is reported. (5) EVo's temporary run files are kept in the system temporary folder, not in the working directory.
